# Jordanian Electricity Bill Explainer — starter notebook

**Pipeline:** bill image → **Qwen2.5-VL** reads the bill → **calculator** applies the EMRC tariff → **Qwen** explains in Arabic.

**Before you start (Colab):** `Runtime → Change runtime type → T4 GPU`. On Kaggle: `Settings → Accelerator → GPU T4`.

Sections:
1. Install
2. The calculator (no GPU needed) + checks against EMRC's official examples
3. Load Qwen2.5-VL
4. Read a bill image → JSON fields
5. Connect: image → fields → calculator → Arabic explanation
6. Evaluate on all bills (`data/bills.csv`)

⚠️ Only use bill images that are already **redacted** (name, account no., meter no., address, barcodes blacked out).

## 1. Install

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils
# Colab already has pandas & pillow - upgrading them breaks Colab's own packages.

### Get the project files
Clones the GitHub repo so `data/bills.csv` and the redacted bill images are available. Change `GITHUB_USER` to your username.

If the repo is **private**, create a token (GitHub → Settings → Developer settings → Personal access tokens) and use
`https://<TOKEN>@github.com/...` as the URL — never commit the token.

In [ ]:
import os

GITHUB_USER = "YOUR-GITHUB-USERNAME"   # <- change this
REPO = "fatoorti"

if os.path.exists("/content"):                     # Colab
    if not os.path.exists(f"/content/{REPO}"):
        !git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
    %cd /content/{REPO}
    !git pull -q
print("working in:", os.getcwd())

## 2. The calculator

Official household tariff (EMRC / kahraba.gov.jo):

| Subsidized | fils/kWh | | Unsubsidized | fils/kWh |
|---|---|---|---|---|
| 1–300 | 50 | | 1–1000 | 120 |
| 301–600 | 100 | | >1000 | 150 |
| >600 | 200 | | | |

Fixed monthly discount (subsidized only, not on building service meters): **2.5 JD** for 51–200 kWh, **2 JD** for 201–600 kWh, none otherwise.

**Extra fees found on our 10 real bills** (all 10 reproduced exactly):

| Fee | Value | Source |
|---|---|---|
| Meter rent أجرة العداد | 0.200 JD | every bill |
| Rural fils فلس الريف | 1 fils / kWh | every bill |
| TV fee رسم التلفزيون | 1.000 JD | every bill |
| Waste fee رسم النفايات | Amman: 0.666 + max(1 JD, 5 fils/kWh) · Irbid: 3.000 JD | **inferred from our bills — still needs an official source** |

Bill structure: `bill value (قيمة الفاتورة) = electricity + fees`, then `amount due (القيمة المطلوبة) = bill value − fixed discount − settlement (حساب التسوية)`.

JEPCO also normalises a longer reading period to 30 days (the **محتسبة** reading): billed kWh ≈ actual kWh × 30 / days. *(Inferred from 3 bills.)*

In [ ]:
SUBSIDIZED_TIERS   = [(300, 0.050), (300, 0.100), (float("inf"), 0.200)]   # (size kWh, JD per kWh)
UNSUBSIDIZED_TIERS = [(1000, 0.120), (float("inf"), 0.150)]

def fixed_discount(kwh: float, subsidized: bool, service_meter: bool = False) -> float:
    if not subsidized or service_meter:
        return 0.0
    if 51 <= kwh <= 200:
        return 2.5
    if 201 <= kwh <= 600:
        return 2.0
    return 0.0

def electricity_bill(kwh: float, subsidized: bool = True, service_meter: bool = False) -> dict:
    """Expected energy charge of a Jordanian household bill under the EMRC tariff."""
    if kwh is None or kwh < 0:
        raise ValueError("kwh must be a non-negative number")
    tiers = SUBSIDIZED_TIERS if subsidized else UNSUBSIDIZED_TIERS
    breakdown, remaining, start = [], kwh, 0
    for size, price in tiers:
        used = min(remaining, size)
        if used <= 0:
            break
        breakdown.append({
            "tier": f"{start + 1:g}-{start + used:g} kWh",
            "kwh": used,
            "price_jd_per_kwh": price,
            "cost_jd": round(used * price, 3),
        })
        remaining -= used
        start += used
    energy = round(sum(t["cost_jd"] for t in breakdown), 3)
    discount = fixed_discount(kwh, subsidized, service_meter)
    return {
        "kwh": kwh,
        "tariff": "subsidized" if subsidized else "unsubsidized",
        "breakdown": breakdown,
        "energy_jd": energy,
        "discount_jd": discount,
        "total_jd": round(energy - discount, 3),   # EMRC's published examples use this (energy - discount)
    }

# ---- Extra fees seen on real bills -------------------------------------------
METER_FEE_JD = 0.200
TV_FEE_JD = 1.000
RURAL_FILS_PER_KWH = 0.001

def waste_fee(kwh: float, city: str = "Amman"):
    """INFERRED from our real bills — confirm with an official source before relying on it."""
    city = (city or "").strip().lower()
    if city == "amman":
        return round(0.666 + max(1.0, 0.005 * kwh), 3)
    if city == "irbid":
        return 3.000
    return None   # unknown city -> the caller must pass the printed value

def full_bill(kwh: float, subsidized: bool = True, city: str = "Amman",
              service_meter: bool = False, waste_fee_jd: float = None, settlement_jd: float = 0.0) -> dict:
    """Reproduces a real JEPCO/IDECO household bill line by line."""
    e = electricity_bill(kwh, subsidized, service_meter)
    waste = waste_fee_jd if waste_fee_jd is not None else waste_fee(kwh, city)
    if waste is None:
        raise ValueError(f"No waste-fee rule for city={city!r}; pass waste_fee_jd from the bill")
    fees = {
        "meter_fee_jd": METER_FEE_JD,
        "rural_fils_jd": round(RURAL_FILS_PER_KWH * kwh, 3),
        "tv_fee_jd": TV_FEE_JD,
        "waste_fee_jd": waste,
    }
    bill_value = round(e["energy_jd"] + sum(fees.values()), 3)
    return {
        **e,
        "fees": fees,
        "bill_value_jd": bill_value,                                   # قيمة الفاتورة
        "amount_due_jd": round(bill_value - e["discount_jd"] - settlement_jd, 3),  # القيمة المطلوبة
    }

def normalise_to_30_days(prev_reading: float, current_reading: float, days: int) -> int:
    """INFERRED: JEPCO bills a longer period as if it were 30 days (the 'محتسبة' reading)."""
    return round((current_reading - prev_reading) * 30 / days)

full_bill(450)

### Check: does it reproduce EMRC's own published examples?

In [ ]:
EMRC_EXAMPLES = [  # (kWh, subsidized?, official JD)
    (100, True, 2.50), (200, True, 7.50), (300, True, 13.0), (400, True, 23.0),
    (100, False, 12.0), (200, False, 24.0), (300, False, 36.0), (400, False, 48.0),
]
for kwh, sub, official in EMRC_EXAMPLES:
    got = electricity_bill(kwh, sub)["total_jd"]
    print(f"{kwh:>4} kWh {'sub  ' if sub else 'unsub'}  calc={got:>6.2f}  official={official:>6.2f}  {'✓' if abs(got - official) < 0.01 else '✗'}")

### Check against ALL real bills in `data/bills.csv`
Every row should show ✓. A new bill that shows ✗ means a fee or rule we haven't seen yet — investigate it.

In [ ]:
import pandas as pd

bills = pd.read_csv("data/bills.csv")
ok = lambda a, b: "✓" if abs(a - b) < 0.0015 else "✗"
for _, b in bills.iterrows():
    r = full_bill(b.billed_kwh, b.tariff_type == "subsidized", b.city, settlement_jd=b.settlement_jd)
    print(f"{b.bill_id} {b.city:6} {b.billed_kwh:>4} kWh | electricity {ok(r['energy_jd'], b.consumption_value_jd)}"
          f" | waste {ok(r['fees']['waste_fee_jd'], b.waste_fee_jd)} | discount {ok(r['discount_jd'], b.fixed_discount_jd)}"
          f" | bill value {r['bill_value_jd']:7.3f} vs {b.bill_value_jd:7.3f} {ok(r['bill_value_jd'], b.bill_value_jd)}"
          f" | due {ok(r['amount_due_jd'], b.amount_due_jd)}")

print("\n30-day normalisation (July bills: read 22/07 after 18/06 = 34 days):")
for prev, cur, days, billed in [(1307, 1548, 34, 213), (3319, 4126, 34, 712), (1667, 2090, 34, 373)]:
    print(f"  {cur - prev} kWh over {days} days -> {normalise_to_30_days(prev, cur, days)} (printed {billed})")

## 3. Load Qwen2.5-VL

- `7B` in 4-bit fits a free T4 (~6–7 GB VRAM) and reads documents well.
- If you run out of memory, switch to `3B`.
- First run downloads ~15 GB of weights (a few minutes).

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"   # or "Qwen/Qwen2.5-VL-3B-Instruct"
USE_8BIT = False                            # 8-bit is slower but can read small digits more reliably

quant = (BitsAndBytesConfig(load_in_8bit=True) if USE_8BIT else
         BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                            bnb_4bit_compute_dtype=torch.float16))

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quant, device_map="auto", torch_dtype=torch.float16)

# Pixel budget: higher = sharper digits but more memory. 2048*28*28 works on a T4.
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=2048*28*28)
print("loaded", MODEL_ID)

In [ ]:
from qwen_vl_utils import process_vision_info

def qwen_chat(content: list, max_new_tokens: int = 512) -> str:
    """content = list of {"type": "image", "image": path} / {"type": "text", "text": ...}"""
    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images, videos = process_vision_info(messages)
    inputs = processor(text=[text], images=images, videos=videos,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()

## 4. Read a bill image → JSON

The vision model only **reads**. It never calculates — that's the calculator's job.

In [ ]:
import json,re
EXTRACT_PROMPT = """You are reading a Jordanian household electricity bill (JEPCO or IDECO). Labels are Arabic.

TOP OF THE BILL - meter readings table:
  سابقة = previous reading, محتسبة = calculated reading, حالية = current reading, الكمية المفوترة = billed kWh.

CHARGES TABLE (البيان) - two number columns: دينار (dinars) and فلس (fils, always 3 digits).
Read the rows from TOP to BOTTOM. For each row copy BOTH columns of THAT row only: {"dinar": "22", "fils": "300"}.
Keep a minus sign if printed: {"dinar": "-2", "fils": "000"}.

Return ONLY this JSON object:
{
  "company": "JEPCO" or "IDECO" or null,
  "tariff_type": "subsidized" if it says مدعوم, "unsubsidized" if the price is 0.12 or 120 fils, else null,
  "prev_reading": number (سابقة),
  "calculated_reading": number (محتسبة) or null,
  "current_reading": number (حالية),
  "billed_kwh": number (الكمية المفوترة or كمية الاستهلاك),
  "consumption_value": row 1 قيمة الاستهلاك,
  "fuel_price_diff": row فرق أسعار الوقود,
  "meter_fee": row أجرة العداد,
  "rural_fils": row فلس الريف,
  "tv_fee": row رسم التلفزيون,
  "waste_fee": row رسم النفايات,
  "fixed_discount": row قيمة الخصم الثابت (0 if missing),
  "bill_value": row قيمة الفاتورة,
  "amount_due": row القيمة المطلوبة
}
Use null for anything you cannot read. Never guess, never calculate."""

ROWS = ["consumption_value", "fuel_price_diff", "meter_fee", "rural_fils",
        "tv_fee", "waste_fee", "fixed_discount", "bill_value", "amount_due"]

def to_jd(v):
    """{'dinar': '12', 'fils': '505'} -> 12.505 ; '189' -> 189.0"""
    if isinstance(v, dict):
        if v.get("dinar") in (None, "") or v.get("fils") in (None, ""):
            return None
        d = str(v["dinar"]).strip().replace("−", "-")
        try:
            jd = abs(int(d)) + int(str(v["fils"]).strip()) / 1000
        except ValueError:
            return None
        return round(-jd if d.startswith("-") else jd, 3)
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def parse_json(text: str):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def _close(a, b, tol=0.0015):
    return a is not None and b is not None and abs(a - b) <= tol

def read_fields(f: dict) -> dict:
    """What the model READ, converted to numbers. This is what you score."""
    r = {"company": f.get("company"), "tariff_type": f.get("tariff_type")}
    for k in ["prev_reading", "calculated_reading", "current_reading", "billed_kwh"]:
        r[k] = to_jd(f.get(k))
    for k in ROWS:
        r[k + "_jd"] = to_jd(f.get(k))
    if r["fixed_discount_jd"] is not None:
        r["fixed_discount_jd"] = abs(r["fixed_discount_jd"])
    return r

def repair(r: dict) -> dict:
    """What the SYSTEM uses: fix misread lines using the calculator and the bill's own totals."""
    out = dict(r)
    kwh = r["billed_kwh"]
    if not kwh:
        out["status"] = "cannot read kWh - ask the user to type it"
        return out
    sub = r["tariff_type"] != "unsubsidized"

    # 1. Fixed fees are anchors: if they're wrong, the model shifted rows
    out["rows_aligned"] = _close(r["meter_fee_jd"], METER_FEE_JD) and _close(r["tv_fee_jd"], TV_FEE_JD)

    # 2. Electricity charge and rural fils follow exactly from kWh
    expected = electricity_bill(kwh, sub)["energy_jd"]
    out["consumption_check_ok"] = _close(r["consumption_value_jd"], expected)
    out["consumption_value_jd"] = expected
    out["rural_fils_jd"] = round(RURAL_FILS_PER_KWH * kwh, 3)
    out["meter_fee_jd"], out["tv_fee_jd"] = METER_FEE_JD, TV_FEE_JD
    fuel = r["fuel_price_diff_jd"] if (out["rows_aligned"] and r["fuel_price_diff_jd"] is not None) else 0.0
    out["fuel_price_diff_jd"] = fuel

    # 3. Waste fee = whatever makes the bill add up
    if r["bill_value_jd"] is None:
        out["status"] = "cannot read bill value - waste fee unverified"
        return out
    implied = round(r["bill_value_jd"] - expected - fuel - METER_FEE_JD - out["rural_fils_jd"] - TV_FEE_JD, 3)
    out["waste_fee_check_ok"] = _close(r["waste_fee_jd"], implied)
    if 0 < implied < 15:
        out["waste_fee_jd"] = implied
        out["status"] = "ok" if (out["consumption_check_ok"] and out["waste_fee_check_ok"]) else "repaired"
    else:
        out["status"] = "totals don't add up - ask the user to check the bill"
    return out

def extract_bill_fields(image_path: str) -> dict:
    raw = qwen_chat([{"type": "image", "image": image_path},
                     {"type": "text", "text": EXTRACT_PROMPT}], max_new_tokens=600)
    parsed = parse_json(raw)
    if not parsed:
        return {"read": None, "fields": None, "raw_output": raw}
    r = read_fields(parsed)
    return {"read": r, "fields": repair(r), "raw_output": raw}

Test the reader on one bill:

In [ ]:
IMAGE_PATH = "data/bills/B08.jpg"      # any redacted bill from the repo
# To test a new bill instead, uncomment:
# from google.colab import files; IMAGE_PATH = next(iter(files.upload()))

result = extract_bill_fields(IMAGE_PATH)
print(result["raw_output"])
result["fields"]

## 5. Connect everything: image → fields → calculator → explanation

This is the **tool-use** step: the LLM's numbers come from `electricity_bill()`, never from its own arithmetic.

In [ ]:
EXPLAIN_PROMPT = """أنت مساعد يشرح فواتير الكهرباء المنزلية في الأردن.
استخدم فقط الأرقام الموجودة في البيانات أدناه. لا تحسب أي رقم بنفسك ولا تخترع أسعاراً.
إذا كان المجموع المطبوع يختلف عن المحسوب، قل ذلك بوضوح وقل إن السبب قد يكون بنوداً إضافية في الفاتورة.
إذا سألك المستخدم عن شيء غير موجود في البيانات (مثل سبب زيادة استهلاكه)، قل إنك لا تعرف.

بيانات الفاتورة والحساب:
{data}

سؤال المستخدم: {question}

اكتب جواباً قصيراً وواضحاً باللهجة الأردنية البسيطة."""

def explain_bill(image_path: str, question: str = "ليش فاتورتي هيك؟") -> dict:
    ext = extract_bill_fields(image_path)
    f = ext["fields"]
    if not f or f.get("billed_kwh") is None:
        return {"error": "Could not read kWh from the image. Ask the user to type it.", "raw": ext["raw_output"]}
    if f.get("status", "").startswith(("cannot", "totals")):
        return {"error": f["status"], "read": ext["read"]}

    subsidized = f.get("tariff_type") != "unsubsidized"   # default: subsidized (most Jordanian households)
    city = "Irbid" if f.get("company") == "IDECO" else "Amman"
    calc = full_bill(float(f["billed_kwh"]), subsidized, city, waste_fee_jd=f.get("waste_fee_jd"))

    data = {
        "extracted_from_bill": f,
        "calculation": calc,
        "printed_vs_calculated_bill_value_diff_jd": (round(f["bill_value_jd"] - calc["bill_value_jd"], 3)
                                                     if f.get("bill_value_jd") is not None else None),
    }
    answer = qwen_chat([{"type": "text", "text": EXPLAIN_PROMPT.format(
        data=json.dumps(data, ensure_ascii=False, indent=2), question=question)}], max_new_tokens=400)
    return {"data": data, "answer": answer}

out = explain_bill(IMAGE_PATH, "ليش فاتورتي هالشهر أعلى من الشهر الماضي؟")
print(json.dumps(out["data"], ensure_ascii=False, indent=2) if "data" in out else out)
print("\n--- Answer ---\n", out.get("answer"))

## 6. Evaluate on all your bills

`bills.csv` holds the **true values typed by hand** (the answer key). The **redacted** images live in `data/bills/` (`B01.jpg` … `B10.jpg`), matching the `image` column.

This cell measures **reader accuracy**: how many fields Qwen read correctly on each bill.

In [ ]:
FIELDS = ["billed_kwh", "tariff_type", "consumption_value_jd", "waste_fee_jd",
          "fixed_discount_jd", "bill_value_jd", "amount_due_jd"]

def same(a, t, tol=0.0015):
    if isinstance(t, str):
        return a == t
    try:
        return abs(float(a) - float(t)) <= tol
    except (TypeError, ValueError):
        return False

rows = []
for _, b in bills.iterrows():
    f = extract_bill_fields(b["image"])["read"] or {}
    rows.append({"bill_id": b.bill_id, **{k: same(f.get(k), b[k]) for k in FIELDS}})

res = pd.DataFrame(rows).set_index("bill_id")
display(res)
print("Per-field accuracy:\n", res.mean().round(2))

## What's next (later phases)

- **Phase 2 baseline:** give a plain LLM the same fields *without* the calculator and score it on the same test set.
- **Phase 3:** add RAG over the EMRC documents so explanations cite the rule, and turn `electricity_bill` into a proper function-calling tool.
- **Phase 4:** compare readers (Qwen 3B vs 7B vs 8-bit vs Gemini) using section 6.